# 实践基础

本章过一遍 PyTorch 最常用的工具：Tensor、autograd、`nn.Module`、`Dataset`/`DataLoader`、模型保存与加载。

## 1. Tensor 基础

In [ ]:
import torch

# 创建
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.zeros(2, 3)
c = torch.ones(2, 3)
d = torch.randn(2, 3)            # 标准正态
e = torch.arange(0, 10, 2)

print(a, a.shape, a.dtype)
print(d)
print(e)

In [ ]:
# 属性与类型转换
x = torch.randn(3, 4)
print('shape :', x.shape)
print('dtype :', x.dtype)
print('device:', x.device)

x_int = x.to(torch.long)
x_cuda = x.to('cuda') if torch.cuda.is_available() else x
print(x_int.dtype, x_cuda.device)

## 2. 数学运算与广播

PyTorch 广播规则与 NumPy 完全一致：从最右维往左对齐，对应维度若相等或其中一边是 1 就可以广播。

In [ ]:
x = torch.tensor([[1.0, 2.0, 3.0]])    # shape (1, 3)
y = torch.tensor([[10.0], [20.0]])     # shape (2, 1)
print(x + y)                            # 广播为 (2, 3)

# 矩阵乘法
A = torch.randn(3, 4)
B = torch.randn(4, 5)
print('A @ B shape:', (A @ B).shape)

# 沿维度归约
M = torch.arange(12).reshape(3, 4).float()
print('sum  :', M.sum())                       # 全部
print('sum0 :', M.sum(dim=0))                  # 沿第 0 维
print('mean1:', M.mean(dim=1, keepdim=True))   # 保留维度

## 3. 形状操作

常用：`view` / `reshape` / `permute` / `transpose` / `squeeze` / `unsqueeze` / `cat` / `stack`。

In [ ]:
x = torch.arange(24)
y = x.view(2, 3, 4)
print('view shape:', y.shape)

z = y.permute(2, 0, 1)               # 维度重排
print('permute shape:', z.shape)

a = torch.zeros(1, 3, 1)
print('squeeze:', a.squeeze().shape)        # 去掉所有为 1 的维
print('unsqueeze:', a.unsqueeze(0).shape)

u = torch.zeros(2, 3)
v = torch.ones(2, 3)
print('cat dim=0:', torch.cat([u, v], dim=0).shape)
print('stack dim=0:', torch.stack([u, v], dim=0).shape)

## 4. 自动微分

标量 loss 调 `.backward()`，梯度沉淀到叶子节点的 `.grad`。中间张量默认不保留梯度，要看的话用 `tensor.retain_grad()` 或 `torch.autograd.grad`。

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = (x ** 2).sum()
y.backward()
print('dy/dx =', x.grad)            # 期望 [2., 4., 6.]

# 反向之前如果不清零，下一次的梯度会累加
x.grad.zero_()

# 推断阶段关掉 autograd 省显存
with torch.no_grad():
    pred = x * 2
print('pred requires_grad:', pred.requires_grad)

# detach：切断计算图，但共享存储
z = (x * 3).detach()
print('z requires_grad:', z.requires_grad)

## 5. `nn.Module` 与一个最小训练循环

In [ ]:
import torch.nn as nn
import torch.optim as optim

class TinyMLP(nn.Module):
    def __init__(self, in_dim=2, hid=16, out=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid),
            nn.ReLU(),
            nn.Linear(hid, out),
        )
    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
# 合成回归数据 y = sum(x) + 噪声
X = torch.randn(256, 2)
y = X.sum(dim=1, keepdim=True) + 0.1 * torch.randn(256, 1)

model = TinyMLP()
opt   = optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

for epoch in range(200):
    pred = model(X)
    loss = loss_fn(pred, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if epoch % 50 == 0:
        print(f'epoch {epoch:3d}  loss {loss.item():.4f}')

## 6. `Dataset` 与 `DataLoader`

自定义数据集只要实现 `__len__` 和 `__getitem__`，剩下交给 `DataLoader` 做批与打乱。

In [ ]:
from torch.utils.data import Dataset, DataLoader

class XOR(Dataset):
    """4 个样本的玩具数据集。"""
    def __init__(self):
        self.X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
        self.y = torch.tensor([0, 1, 1, 0])
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i]

ds = XOR()
loader = DataLoader(ds, batch_size=2, shuffle=True)
for x, y in loader:
    print(x, y)

## 7. 模型的保存与加载

推荐保存 `state_dict`，而不是整个 model 对象——前者跨结构改动更稳。

In [ ]:
torch.save(model.state_dict(), 'tinymlp.pt')

model2 = TinyMLP()
model2.load_state_dict(torch.load('tinymlp.pt'))
model2.eval()

# 验证：两份模型在同样输入下输出应当一致
with torch.no_grad():
    a = model(X[:5])
    b = model2(X[:5])
print('max abs diff:', (a - b).abs().max().item())

import os; os.remove('tinymlp.pt')